In [0]:
%sql
CREATE OR REPLACE TABLE simo_show_case_catalog.lakeview_ml.note_social_risk
COMMENT 'GenAI (ai_extract v2.1) social determinants of health extracted from discharge summaries, one row per encounter. Missing mentions are coalesced to false.'
AS
WITH extracted AS (
  SELECT encounter_id, patient_id, note_id,
    ai_extract(note_text, '{
      "lives_alone":             {"type": "boolean", "description": "True ONLY if the note explicitly says the patient lives alone / by themselves or has no one at home. Being independent with activities of daily living does NOT mean living alone."},
      "caregiver_available":     {"type": "boolean", "description": "A family member or caregiver is explicitly available at home to help after discharge"},
      "transportation_barrier":  {"type": "boolean", "description": "Patient lacks reliable transportation to appointments or the pharmacy"},
      "medication_cost_barrier": {"type": "boolean", "description": "Patient cannot afford, rations, or skips medications because of cost"},
      "health_literacy_concern": {"type": "boolean", "description": "Difficulty understanding instructions, incomplete teach-back, or an interpreter was needed"},
      "substance_use":           {"type": "boolean", "description": "Current tobacco, heavy alcohol, or drug use, OR any history of a substance use disorder (e.g. opioid use disorder) even if not currently in treatment"},
      "social_risk_summary":     {"type": "string",  "description": "One short phrase summarizing the social risks for a care coordinator, or none"}
    }', MAP('version', '2.1',
            'instructions', 'These are hospital discharge summaries. Extract social determinants of health from the Social history section only. Return false when a factor is not mentioned.')) AS x
  FROM simo_show_case_catalog.lakeview_health.silver_discharge_notes
)
SELECT encounter_id, patient_id, note_id,
  coalesce(x:response.lives_alone.value::BOOLEAN, false)             AS sdoh_lives_alone,
  coalesce(x:response.caregiver_available.value::BOOLEAN, false)     AS sdoh_caregiver_available,
  coalesce(x:response.transportation_barrier.value::BOOLEAN, false)  AS sdoh_transportation_barrier,
  coalesce(x:response.medication_cost_barrier.value::BOOLEAN, false) AS sdoh_medication_cost_barrier,
  coalesce(x:response.health_literacy_concern.value::BOOLEAN, false) AS sdoh_health_literacy_concern,
  coalesce(x:response.substance_use.value::BOOLEAN, false)           AS sdoh_substance_use,
  x:response.social_risk_summary.value::STRING                       AS social_risk_summary,
  x:error_message::STRING                                            AS extraction_error,
  current_timestamp()                                                AS extracted_at
FROM extracted;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS notes, count_if(extraction_error IS NOT NULL) AS errors,
  round(avg(sdoh_lives_alone::INT), 3) AS lives_alone, round(avg(sdoh_caregiver_available::INT), 3) AS caregiver,
  round(avg(sdoh_transportation_barrier::INT), 3) AS transport, round(avg(sdoh_medication_cost_barrier::INT), 3) AS med_cost,
  round(avg(sdoh_health_literacy_concern::INT), 3) AS literacy, round(avg(sdoh_substance_use::INT), 3) AS substance
FROM simo_show_case_catalog.lakeview_ml.note_social_risk

notes,errors,lives_alone,caregiver,transport,med_cost,literacy,substance
13200,0,0.196,0.547,0.104,0.088,0.087,0.088


In [0]:
%sql
-- Independent check: compare ai_extract with simple keyword rules on the Social history text.
-- High agreement shows the extraction works. Disagreements show where the LLM handles phrasing that rules miss.
WITH rules AS (
  SELECT n.encounter_id,
    lower(n.note_text) RLIKE 'lives alone|living alone|lives by (him|her)self'           AS r_lives_alone,
    lower(n.note_text) RLIKE 'transportation|does not drive|relies on the bus|no car'     AS r_transport,
    lower(n.note_text) RLIKE 'afford|rationing|because of cost|financial strain'          AS r_med_cost,
    lower(n.note_text) RLIKE 'teach-back incomplete|health literacy|interpreter|unsure how to use' AS r_literacy,
    lower(n.note_text) RLIKE 'current smoker|daily alcohol|opioid use disorder'           AS r_substance
  FROM simo_show_case_catalog.lakeview_health.silver_discharge_notes n
)
SELECT 'lives_alone' AS factor, round(avg((s.sdoh_lives_alone = r.r_lives_alone)::INT), 4) AS agreement,
       count_if(s.sdoh_lives_alone AND NOT r.r_lives_alone) AS genai_only, count_if(r.r_lives_alone AND NOT s.sdoh_lives_alone) AS rules_only
FROM simo_show_case_catalog.lakeview_ml.note_social_risk s JOIN rules r USING (encounter_id)
UNION ALL SELECT 'transportation', round(avg((s.sdoh_transportation_barrier = r.r_transport)::INT), 4), count_if(s.sdoh_transportation_barrier AND NOT r.r_transport), count_if(r.r_transport AND NOT s.sdoh_transportation_barrier)
FROM simo_show_case_catalog.lakeview_ml.note_social_risk s JOIN rules r USING (encounter_id)
UNION ALL SELECT 'medication_cost', round(avg((s.sdoh_medication_cost_barrier = r.r_med_cost)::INT), 4), count_if(s.sdoh_medication_cost_barrier AND NOT r.r_med_cost), count_if(r.r_med_cost AND NOT s.sdoh_medication_cost_barrier)
FROM simo_show_case_catalog.lakeview_ml.note_social_risk s JOIN rules r USING (encounter_id)
UNION ALL SELECT 'health_literacy', round(avg((s.sdoh_health_literacy_concern = r.r_literacy)::INT), 4), count_if(s.sdoh_health_literacy_concern AND NOT r.r_literacy), count_if(r.r_literacy AND NOT s.sdoh_health_literacy_concern)
FROM simo_show_case_catalog.lakeview_ml.note_social_risk s JOIN rules r USING (encounter_id)
UNION ALL SELECT 'substance_use', round(avg((s.sdoh_substance_use = r.r_substance)::INT), 4), count_if(s.sdoh_substance_use AND NOT r.r_substance), count_if(r.r_substance AND NOT s.sdoh_substance_use)
FROM simo_show_case_catalog.lakeview_ml.note_social_risk s JOIN rules r USING (encounter_id)

factor,agreement,genai_only,rules_only
lives_alone,0.994,3,76
transportation,0.9998,2,1
medication_cost,1.0,0,0
health_literacy,1.0,0,0
substance_use,0.9998,2,0


### Prompt iteration (AI as a teammate, checked against evidence)
**v1 schema results (first run):** agreement with the keyword rules was 91.8% for lives_alone and 98.0% for substance_use. Inspecting the disagreements showed two systematic errors:
* 715 notes saying *"Independent with activities of daily living"* were wrongly tagged **lives_alone = true**, a false positive.
* *"History of opioid use disorder, not currently in treatment"* was tagged **substance_use = false**, a false negative that matters clinically.

**Fix (v2, the version in the cell above):** tightened the field descriptions. lives_alone now requires an explicit statement and says that being independent with daily activities does not count. substance_use now includes any history of substance use disorder. The validation cell re-measures agreement after the change.

**v2 results (validation cell above):** agreement rose to **99.4%** for lives_alone (genai-only false positives fell from 988 to 3) and **99.98%** for substance_use (the 260 misses went to 0). The signal also got cleaner: patients who live alone are readmitted at 25.2% vs 13.1% (v1: 22.3% vs 13.0%).

In [0]:
%sql
-- Do the GenAI-extracted factors carry signal? Compare the readmission rate with and without each factor.
SELECT factor, round(avg(CASE WHEN flag THEN y END), 4) AS rate_with_factor, round(avg(CASE WHEN NOT flag THEN y END), 4) AS rate_without_factor, count_if(flag) AS n_with_factor
FROM (
  SELECT a.readmitted_30d::INT AS y, stack(5,
    'lives_alone', s.sdoh_lives_alone, 'transportation', s.sdoh_transportation_barrier, 'medication_cost', s.sdoh_medication_cost_barrier,
    'health_literacy', s.sdoh_health_literacy_concern, 'substance_use', s.sdoh_substance_use) AS (factor, flag)
  FROM simo_show_case_catalog.lakeview_health.gold_index_admissions a
  JOIN simo_show_case_catalog.lakeview_ml.note_social_risk s USING (encounter_id)
  WHERE a.label_window_complete AND a.is_measure_eligible
)
GROUP BY factor ORDER BY rate_with_factor DESC

factor,rate_with_factor,rate_without_factor,n_with_factor
lives_alone,0.2522,0.1314,2435
transportation,0.2513,0.1437,1305
medication_cost,0.2406,0.1467,1093
health_literacy,0.1902,0.1516,1099
substance_use,0.1895,0.1518,1050
